# 04 — Exploration vs. Exploitation

Analyses for the SMM2 explore/exploit paradigm:

1. Exploration rate by run and block condition
2. Exploration decay within blocks (information accumulation)
3. Reward sensitivity (EV=0 vs EV=8 comparison)
4. Model fitting: **ε-greedy**, **UCB1**, **Softmax**
5. Directed vs. random exploration classification

**Block conditions** (from spec):
- Block 1, runs 1–5:   explore reward = **0** (EV=0)
- Block 2, runs 6–10:  explore reward = **8** (EV=8)
- Block 3, runs 11–15: explore reward = **3** (EV=3, tied)
- Block 4, runs 16–20: explore reward = **0** (EV=0, re-test)

Run `00_data_pipeline` first.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.optimize import minimize
from scipy.special import expit  # sigmoid

ROOT     = Path('..').resolve()
PROC_DIR = ROOT / 'data' / 'processed'
FIG_DIR  = ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})

def load(name):
    p = PROC_DIR / f'{name}.parquet'
    return pd.read_parquet(p) if p.exists() else pd.DataFrame()

# Block definitions from spec
BLOCK_MAP = {
    1: {'runs': range(1,  6),  'explore_reward': 0, 'label': 'EV=0 (initial)'},
    2: {'runs': range(6,  11), 'explore_reward': 8, 'label': 'EV=8 (high)'},
    3: {'runs': range(11, 16), 'explore_reward': 3, 'label': 'EV=3 (tied)'},
    4: {'runs': range(16, 21), 'explore_reward': 0, 'label': 'EV=0 (re-test)'},
}
EXPLOIT_REWARD_BP1 = 4   # known reward at branch point 1
EXPLOIT_REWARD_BP2 = 2   # known reward at branch point 2


## 1. Build run-level choice dataset


In [ ]:
ee = load('explore_exploit')

if not ee.empty:
    # Extract run number from Note field on BRANCH_POINT events
    bp = ee[ee['event_code'] == 'BRANCH_POINT'].copy()
    bp['run_num'] = pd.to_numeric(
        bp['note'].str.extract(r'run\s*(\d+)', expand=False), errors='coerce')
    bp['bp_num'] = bp['note'].str.extract(r'BP(\d+)', expand=False).fillna('1')

    # Join choices (EXPLORE/EXPLOIT) to each BRANCH_POINT event
    choices = ee[ee['event_code'].isin(['EXPLORE', 'EXPLOIT'])].copy()

    # Merge: for each participant/session/elapsed range after BRANCH_POINT,
    # find the next choice
    choice_rows = []
    for (pid, sid), grp in ee.groupby(['participant_id', 'session_id']):
        grp = grp.sort_values('elapsed_s').reset_index(drop=True)
        run_num = None
        for i, row in grp.iterrows():
            if row['event_code'] == 'BRANCH_POINT':
                try:
                    run_num = int(
                        pd.Series([row['note']]).str.extract(r'run\s*(\d+)')[0].iloc[0])
                except Exception:
                    pass
                bp_num = row['note'].strip()[-1] if row['note'].strip() else '1'
            elif row['event_code'] in ('EXPLORE', 'EXPLOIT') and run_num is not None:
                reward_code = None
                # Look ahead for REWARD or NO_REWARD
                future = grp.iloc[i+1:i+5]
                if 'REWARD' in future['event_code'].values:
                    reward_code = 'REWARD'
                elif 'NO_REWARD' in future['event_code'].values:
                    reward_code = 'NO_REWARD'

                # Determine block
                block_num = next(
                    (b for b, bd in BLOCK_MAP.items() if run_num in bd['runs']), None)

                choice_rows.append({
                    'participant_id': pid,
                    'session_id':     sid,
                    'run_num':        run_num,
                    'block_num':      block_num,
                    'elapsed_s':      row['elapsed_s'],
                    'choice':         row['event_code'],   # EXPLORE or EXPLOIT
                    'explored':       int(row['event_code'] == 'EXPLORE'),
                    'reward_outcome': reward_code,
                    'explore_reward': BLOCK_MAP.get(block_num, {}).get('explore_reward', np.nan),
                })

    if choice_rows:
        ch = pd.DataFrame(choice_rows)
        ch['run_in_block'] = ch.groupby(['participant_id', 'block_num'])['run_num'] \
                               .transform(lambda x: x - x.min() + 1)
        print(f'Choice events: {len(ch)}')
        display(ch.head(10))
    else:
        print('No choice data. Check that BRANCH_POINT notes include "run N" and "BP1/BP2".')


## 2. Exploration rate by run and block


In [ ]:
if 'ch' in dir() and not ch.empty:
    run_rate = (ch.groupby(['participant_id', 'block_num', 'run_num', 'run_in_block'])
                  .agg(explore_rate=('explored', 'mean'),
                       n_choices=('explored', 'count'))
                  .reset_index())

    fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
    block_colors = {1: '#e63946', 2: '#2a9d8f', 3: '#f4a261', 4: '#457b9d'}

    for ax, (bnum, bdata) in zip(axes, BLOCK_MAP.items()):
        bdf = run_rate[run_rate['block_num'] == bnum]
        for pid, pgrp in bdf.groupby('participant_id'):
            pgrp = pgrp.sort_values('run_in_block')
            ax.plot(pgrp['run_in_block'], pgrp['explore_rate'],
                    marker='o', ms=6, label=pid,
                    color=block_colors.get(bnum, 'gray'))
        ax.axhline(0.5, ls='--', color='gray', alpha=0.4, lw=1)
        ax.set_xlabel('Run within block')
        ax.set_title(bdata['label'])
        ax.set_ylim(-0.05, 1.05)
        ax.set_xticks(range(1, 6))
        if ax is axes[0]:
            ax.set_ylabel('Exploration rate')
        ax.legend(fontsize=8)

    fig.suptitle('Exploration Rate by Run and Block Condition', fontsize=13)
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'explore_exploit_rate_by_run.png', bbox_inches='tight')
    plt.show()


## 3. Reward sensitivity — EV=0 vs EV=8


In [ ]:
if 'ch' in dir() and not ch.empty:
    sensitivity = (ch[ch['block_num'].isin([1, 2])]
                   .groupby(['participant_id', 'block_num'])
                   .agg(explore_rate=('explored', 'mean'))
                   .reset_index())
    sensitivity['block_label'] = sensitivity['block_num'].map(
        {1: 'EV=0 (block 1)', 2: 'EV=8 (block 2)'})

    pivot = sensitivity.pivot(index='participant_id',
                               columns='block_label',
                               values='explore_rate')
    print('Reward sensitivity (EV=0 vs EV=8 exploration rates):')
    display(pivot.round(3))

    if len(pivot) >= 2:
        from scipy.stats import ttest_rel
        t, p = ttest_rel(pivot.iloc[:, 0], pivot.iloc[:, 1])
        print(f'\nPaired t-test: t = {t:.3f}, p = {p:.4f}')
        print('(Significant p < .05 indicates reward sensitivity)')


## 4. Model fitting

### ε-greedy
At each trial, choose randomly with probability ε; exploit known-best with 1−ε.

### UCB1
Choose option i maximising $Q(i) + c\sqrt{\ln n / n_i}$ where $c$ is the exploration bonus.

### Softmax (temperature)
$P(\text{explore}) = \sigma((Q_{\text{explore}} - Q_{\text{exploit}}) / T)$
where $T$ is temperature and $\sigma$ is the logistic function.


In [ ]:
def q_estimate(choices_so_far, rewards_so_far, option):
    """Simple sample-mean Q estimate for one option."""
    idx = [i for i, c in enumerate(choices_so_far) if c == option]
    if not idx:
        return 0.0
    return np.mean([rewards_so_far[i] for i in idx])


def neg_log_likelihood_epsilon(eps, choices, rewards):
    """NLL for epsilon-greedy model."""
    eps = float(np.clip(eps, 1e-6, 1 - 1e-6))
    nll = 0.0
    Q = {0: 0.0, 1: 0.0}   # 0=exploit, 1=explore
    counts = {0: 0, 1: 0}
    for c, r in zip(choices, rewards):
        best = max(Q, key=Q.get)
        p_explore = eps / 2 + (1 - eps) * (1 if best == 1 else 0)
        p = p_explore if c == 1 else 1 - p_explore
        nll -= np.log(max(p, 1e-9))
        # Update Q
        counts[c] += 1
        Q[c] += (r - Q[c]) / counts[c]
    return nll


def neg_log_likelihood_softmax(temp, choices, rewards, q_exploit=4.0):
    """NLL for softmax model. q_exploit = known exploit reward."""
    T = float(np.clip(temp, 1e-4, 50))
    nll = 0.0
    Q_explore = 0.0
    counts_explore = 0
    for c, r in zip(choices, rewards):
        diff = (Q_explore - q_exploit) / T
        p_explore = expit(diff)
        p = p_explore if c == 1 else 1 - p_explore
        nll -= np.log(max(p, 1e-9))
        if c == 1:
            counts_explore += 1
            Q_explore += (r - Q_explore) / counts_explore
    return nll


if 'ch' in dir() and not ch.empty:
    model_results = []

    for (pid, bnum), grp in ch.groupby(['participant_id', 'block_num']):
        grp = grp.sort_values('run_num')
        choices_seq = grp['explored'].tolist()
        # Observed explore reward (0 if no reward outcome, actual if known)
        rewards_seq = [
            grp['explore_reward'].iloc[0] if (r == 1 and o == 'REWARD') else 0
            for r, o in zip(grp['explored'], grp['reward_outcome'].fillna(''))
        ]
        if len(choices_seq) < 3:
            continue

        # --- epsilon-greedy ---
        res_eps = minimize(neg_log_likelihood_epsilon, x0=[0.2],
                           args=(choices_seq, rewards_seq),
                           bounds=[(1e-6, 1 - 1e-6)], method='L-BFGS-B')
        eps_hat = float(res_eps.x[0])

        # --- softmax ---
        exploit_r = EXPLOIT_REWARD_BP1  # use BP1 known reward as baseline
        res_sfx = minimize(neg_log_likelihood_softmax, x0=[2.0],
                           args=(choices_seq, rewards_seq, exploit_r),
                           bounds=[(1e-4, 50)], method='L-BFGS-B')
        temp_hat = float(res_sfx.x[0])

        model_results.append({
            'participant_id': pid,
            'block_num':      bnum,
            'block_label':    BLOCK_MAP[bnum]['label'],
            'explore_rate':   np.mean(choices_seq),
            'epsilon_hat':    round(eps_hat,  4),
            'temperature_hat':round(temp_hat, 4),
            'nll_epsilon':    round(res_eps.fun, 3),
            'nll_softmax':    round(res_sfx.fun, 3),
        })

    if model_results:
        mod_df = pd.DataFrame(model_results)
        print('=== Model Parameter Estimates ===')
        display(mod_df)
        print('\nNotes:')
        print('  epsilon_hat: estimated random exploration rate (ε-greedy)')
        print('  temperature_hat: decision noise (softmax); higher T = more random')
        print('  Compare nll_epsilon vs nll_softmax to assess model fit (lower = better)')


## 5. Directed vs. random exploration

Following Wilson et al. (2021): **run 1** of each block is the directed
exploration proxy (uncertainty is highest); **runs 3–5** are the random
exploration proxy (uncertainty resolved).


In [ ]:
if 'ch' in dir() and not ch.empty:
    ch['explore_type'] = np.where(
        ch['run_in_block'] == 1, 'directed_proxy',
        np.where(ch['run_in_block'] >= 3, 'random_proxy', 'transitional'))

    directed_vs_random = (
        ch[ch['explore_type'].isin(['directed_proxy', 'random_proxy'])]
        .groupby(['participant_id', 'block_num', 'explore_type'])
        .agg(explore_rate=('explored', 'mean'))
        .reset_index()
    )

    pivot_dvr = directed_vs_random.pivot_table(
        index=['participant_id', 'block_num'],
        columns='explore_type', values='explore_rate')
    pivot_dvr.columns.name = None
    display(pivot_dvr.round(3))

    # Bar chart: directed vs random by block
    if 'directed_proxy' in pivot_dvr.columns and 'random_proxy' in pivot_dvr.columns:
        fig, ax = plt.subplots(figsize=(9, 4))
        blocks = sorted(ch['block_num'].unique())
        x = np.arange(len(blocks))
        w = 0.35

        dir_means  = [directed_vs_random[
            (directed_vs_random['block_num'] == b) &
            (directed_vs_random['explore_type'] == 'directed_proxy')
        ]['explore_rate'].mean() for b in blocks]
        rand_means = [directed_vs_random[
            (directed_vs_random['block_num'] == b) &
            (directed_vs_random['explore_type'] == 'random_proxy')
        ]['explore_rate'].mean() for b in blocks]

        ax.bar(x - w/2, dir_means,  w, label='Directed (run 1)',  color='#457b9d', alpha=0.85)
        ax.bar(x + w/2, rand_means, w, label='Random (runs 3–5)', color='#e63946', alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels([BLOCK_MAP[b]['label'] for b in blocks], rotation=15)
        ax.set_ylabel('Exploration rate')
        ax.set_ylim(0, 1.1)
        ax.set_title('Directed vs. Random Exploration by Block')
        ax.legend()
        fig.savefig(FIG_DIR / 'explore_exploit_directed_vs_random.png', bbox_inches='tight')
        plt.show()

        print('\nInterpretation (Wilson et al., 2021):')
        print('  directed_proxy > random_proxy → directed exploration present')
        print('  directed_proxy ≈ random_proxy → predominantly random exploration')
